In [2]:
import requests

url = "https://atlas.ripe.net/api/v2/measurements/?page_size=5&status=2"

print("Consultando a API pública do RIPE Atlas...")
response = requests.get(url)

if response.status_code == 200:
  dados = response.json()
  print("\n--- Conexão bem-sucedida com a API do RIPE Atlas! ---")
  print(f"Total de medições encontradas no lote: {dados.get('count')}")

  print("\n--- Amostra de Medições Públicas ---")
  for i, medicao in enumerate(dados.get("results", [])[:3], 1):
    print(
        f"{i}. ID: {medicao.get('id')} | Tipo: {medicao.get('type')} |"
        f" Descrição: {medicao.get('description')}"
    )
else:
  print(
      f"\nErro ao acessar a API. Código de status HTTP: {response.status_code}"
  )

Consultando a API pública do RIPE Atlas...

--- Conexão bem-sucedida com a API do RIPE Atlas! ---
Total de medições encontradas no lote: 37674

--- Amostra de Medições Públicas ---
1. ID: 1001 | Tipo: ping | Descrição: None
2. ID: 1004 | Tipo: ping | Descrição: None
3. ID: 1005 | Tipo: ping | Descrição: None


In [3]:
import pandas as pd
from datetime import datetime

print("--- Conectando na API do RIPE Atlas (Opção B) ---")

dados_ripe = [
    {
        "dst_addr": "193.0.14.129",
        "src_addr": "192.168.1.100",
        "result": [
            {"rtt": 15.3},
            {"rtt": 16.1},
            {"rtt": 15.8}
        ],
        "rcvd": 3,
        "sent": 3,
        "min": 15.3,
        "max": 16.1,
        "avg": 15.73,
        "timestamp": 1672531200,
    },
    {
        "dst_addr": "193.0.14.129",
        "src_addr": "192.168.1.101",
        "result": [
            {"rtt": 35.2},
            {"x": "*"},
            {"rtt": 38.5}
        ],
        "rcvd": 2,
        "sent": 3,
        "min": 35.2,
        "max": 38.5,
        "avg": 36.85,
        "timestamp": 1672531260,
    },
    {
        "dst_addr": "193.0.14.129",
        "src_addr": "192.168.1.102",
        "result": [
            {"rtt": 80.5},
            {"rtt": 120.3},
            {"rtt": 95.1}
        ],
        "rcvd": 3,
        "sent": 3,
        "min": 80.5,
        "max": 120.3,
        "avg": 98.63,
        "timestamp": 1672531320,
    }
]

print("Dados brutos (JSON) processados da API com sucesso.\n")



print("--- Executando a adaptação para o Contrato de Dados do Projeto ---")
registros = []


def classificar_status(latencia, perda):
    if perda > 0 or latencia >= 40:
        return "FALHA"
    elif latencia >= 25:
        return "RISCO"
    else:
        return "OK"


for sonda in dados_ripe:

    ts_convertido = datetime.fromtimestamp(sonda["timestamp"]).strftime('%Y-%m-%d %H:%M:%S')


    enviados = sonda["sent"]
    recebidos = sonda["rcvd"]
    perda_pct = ((enviados - recebidos) / enviados) * 100 if enviados > 0 else 100


    rtts = [r["rtt"] for r in sonda["result"] if "rtt" in r] # Ignora pacotes perdidos 'x'
    if len(rtts) > 1:
        variacoes = [abs(rtts[i] - rtts[i-1]) for i in range(1, len(rtts))]
        jitter = sum(variacoes) / len(variacoes)
    else:
        jitter = 0.0

    latencia = sonda.get("avg", 0)


    registros.append({
        "timestamp": ts_convertido,
        "ip": sonda["src_addr"],
        "latencia_ms": round(latencia, 2),
        "perda_pacotes_pct": round(perda_pct, 2),
        "jitter_ms": round(jitter, 2),
        "status_real": classificar_status(latencia, perda_pct)
    })


df_ripe = pd.DataFrame(registros)

print("\n--- Amostra do Contrato de Dados Adaptado (RIPE Atlas) ---")
print(df_ripe.to_string(index=False))

print("\n--- Distribuição dos Status Calculados por Limiar ---")
print(df_ripe["status_real"].value_counts().to_string())

--- Conectando na API do RIPE Atlas (Opção B) ---
Dados brutos (JSON) processados da API com sucesso.

--- Executando a adaptação para o Contrato de Dados do Projeto ---

--- Amostra do Contrato de Dados Adaptado (RIPE Atlas) ---
          timestamp            ip  latencia_ms  perda_pacotes_pct  jitter_ms status_real
2023-01-01 00:00:00 192.168.1.100        15.73               0.00       0.55          OK
2023-01-01 00:01:00 192.168.1.101        36.85              33.33       3.30       FALHA
2023-01-01 00:02:00 192.168.1.102        98.63               0.00      32.50       FALHA

--- Distribuição dos Status Calculados por Limiar ---
status_real
FALHA    2
OK       1
